In [ ]:
import yfinance as yf

df = yf.download('005930.KS', start='20-01-01', end='2025-08-01')

# CSV로 저장
df.to_csv('samsung_stock.csv')


/tmp/ipykernel_164437/3418726838.py:4: FutureWarning: YF.download() has changed argument auto_adjust default to True
  df = yf.download('005930.KS', start='2024-01-01', end='2025-08-01')
[*********************100%***********************]  1 of 1 completed


In [9]:
import oracledb
import pandas as pd

df = pd.read_csv("samsung_stock.csv")

# 날짜 포맷 명시적으로 지정
df['Price'] = pd.to_datetime(df['Price'], format='%Y-%m-%d', errors='coerce')
df = df.dropna(subset=['Price'])
df['Price'] = df['Price'].dt.strftime('%Y-%m-%d')

conn = oracledb.connect(
    user="app",
    password="1234!",
    dsn="localhost:1521/XEPDB1"
)
cursor = conn.cursor()

for index, row in df.iterrows():
    cursor.execute(
        """
        INSERT INTO stock_price (
            stock_code, stock_name, trade_date, open_price, high_price,
            low_price, close_price, volume
        ) VALUES (:1, :2, TO_DATE(:3, 'YYYY-MM-DD'), :4, :5, :6, :7, :8)
        """,
        (
            '005930',         # stock_code
            '삼성전자',        # stock_name
            row['Price'],     # 'YYYY-MM-DD' 문자열
            row['Open'],
            row['High'],
            row['Low'],
            row['Close'],
            row['Volume']
        )
    )

conn.commit()
cursor.close()
conn.close()
